In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

data_folder = r'C:\Users\paulc\OneDrive\Projects\wimbledon-predictor\data'

df = pd.read_csv(f'{data_folder}\\grass_combined_2000_2025.csv', 
                 parse_dates=['tourney_date'])
df = df.sort_values('tourney_date').reset_index(drop=True)
print(f"Loaded: {df.shape}")
print(df.head())

Loaded: (7593, 11)
       tourney_name tourney_date surface      round    winner_name  \
0     stella artois   2000-06-12   grass  2nd round      llodra m.   
1  gerry weber open   2000-06-12   grass  2nd round  kafelnikov y.   
2  gerry weber open   2000-06-12   grass  1st round     larsson m.   
3     stella artois   2000-06-12   grass  1st round      llodra m.   
4  gerry weber open   2000-06-12   grass  1st round    burrieza o.   

     loser_name  winner_rank loser_rank  best_of       source  year  
0         sa a.        169.0         84      3.0  tennis_data  2000  
1   burrieza o.          5.0        174      3.0  tennis_data  2000  
2  ferrero j.c.         49.0         12      3.0  tennis_data  2000  
3     rochus c.        169.0         86      3.0  tennis_data  2000  
4      alami k.        174.0         30      3.0  tennis_data  2000  


In [2]:
def compute_features(df_input):
    df = df_input.copy().sort_values('tourney_date').reset_index(drop=True)

    # Initialise columns
    for col in ['winner_elo','loser_elo','winner_grass_win_pct','loser_grass_win_pct',
                'winner_form5_grass','loser_form5_grass','winner_form10_all',
                'loser_form10_all','winner_last_played','loser_last_played']:
        df[col] = np.nan

    # State trackers
    elo         = defaultdict(lambda: 1500.0)
    grass_wins  = defaultdict(int)
    grass_games = defaultdict(int)
    form_grass  = defaultdict(list)
    form_all    = defaultdict(list)
    last_played = defaultdict(lambda: pd.NaT)
    K = 32

    for i, row in df.iterrows():
        w, l   = row['winner_name'], row['loser_name']
        date   = row['tourney_date']

        # Record pre-match state
        df.at[i, 'winner_elo']           = elo[w]
        df.at[i, 'loser_elo']            = elo[l]
        df.at[i, 'winner_grass_win_pct'] = grass_wins[w] / grass_games[w] if grass_games[w] else 0.0
        df.at[i, 'loser_grass_win_pct']  = grass_wins[l] / grass_games[l] if grass_games[l] else 0.0
        df.at[i, 'winner_form5_grass']   = np.mean(form_grass[w][-5:]) if form_grass[w] else 0.0
        df.at[i, 'loser_form5_grass']    = np.mean(form_grass[l][-5:]) if form_grass[l] else 0.0
        df.at[i, 'winner_form10_all']    = np.mean(form_all[w][-10:]) if form_all[w] else 0.0
        df.at[i, 'loser_form10_all']     = np.mean(form_all[l][-10:]) if form_all[l] else 0.0
        df.at[i, 'winner_last_played']   = last_played[w]
        df.at[i, 'loser_last_played']    = last_played[l]

        # Update Elo
        Rw, Rl = elo[w], elo[l]
        Ew = 1 / (1 + 10**((Rl - Rw) / 400))
        elo[w] += K * (1 - Ew)
        elo[l] += K * (0 - (1 - Ew))

        # Update trackers
        grass_wins[w]  += 1
        grass_games[w] += 1
        grass_games[l] += 1
        form_grass[w].append(1); form_grass[l].append(0)
        form_all[w].append(1);   form_all[l].append(0)
        last_played[w] = date;   last_played[l] = date

    return df

print("Running feature engineering — this may take a minute...")
df_feat = compute_features(df)
print(f"Done: {df_feat.shape}")

Running feature engineering — this may take a minute...
Done: (7593, 21)


In [5]:
df_feat['winner_last_played'] = pd.to_datetime(df_feat['winner_last_played'], errors='coerce')
df_feat['loser_last_played']  = pd.to_datetime(df_feat['loser_last_played'], errors='coerce')

df_feat['rest_days_winner'] = (df_feat['tourney_date'] - df_feat['winner_last_played']).dt.days.fillna(0)
df_feat['rest_days_loser']  = (df_feat['tourney_date'] - df_feat['loser_last_played']).dt.days.fillna(0)
df_feat['rest_days_diff']   = df_feat['rest_days_winner'] - df_feat['rest_days_loser']

df_feat['month']     = df_feat['tourney_date'].dt.month
df_feat['month_sin'] = np.sin(2 * np.pi * df_feat['month'] / 12)
df_feat['month_cos'] = np.cos(2 * np.pi * df_feat['month'] / 12)

df_feat['is_wimbledon'] = df_feat['tourney_name'].str.contains('wimbledon', na=False).astype(int)

round_map = {'r128':1,'r64':2,'r32':3,'r16':4,'qf':5,'sf':6,'f':7,'final':7}
df_feat['round_encoded'] = df_feat['round'].map(round_map).fillna(0).astype(int)

df_feat['year'] = df_feat['tourney_date'].dt.year

print("Features computed!")
print(df_feat[['rank_diff','elo_diff','grass_win_pct_diff','form5_grass_diff','is_wimbledon']].head())

Features computed!
   rank_diff  elo_diff  grass_win_pct_diff  form5_grass_diff  is_wimbledon
0      -85.0       0.0                 0.0               0.0             0
1       47.0       0.0                 0.0               0.0             0
2     -188.0       0.0                 0.0               0.0             0
3      -29.0       0.0                 0.0               0.0             0
4      -46.0       0.0                 0.0               0.0             0


In [6]:
feature_cols = [
    'rank_diff', 'elo_diff', 'grass_win_pct_diff',
    'form5_grass_diff', 'form10_all_diff', 'rest_days_diff',
    'month_sin', 'month_cos', 'is_wimbledon', 'round_encoded'
]

# Winner rows — label 1
wins = df_feat.copy()
wins['label'] = 1

# Mirror rows — swap winner/loser, label 0
losses = df_feat.copy()
losses['label'] = 0
losses['rank_diff']          = -losses['rank_diff']
losses['elo_diff']           = -losses['elo_diff']
losses['grass_win_pct_diff'] = -losses['grass_win_pct_diff']
losses['form5_grass_diff']   = -losses['form5_grass_diff']
losses['form10_all_diff']    = -losses['form10_all_diff']
losses['rest_days_diff']     = -losses['rest_days_diff']

balanced = pd.concat([wins, losses], ignore_index=True)
balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)

save_path = f'{data_folder}\\grass_features_balanced.csv'
balanced[feature_cols + ['label', 'year', 'tourney_name', 
                          'winner_name', 'loser_name']].to_csv(save_path, index=False)

print(f"Balanced dataset: {balanced.shape}")
print(f"Label distribution:\n{balanced['label'].value_counts()}")
print(f"Saved to: {save_path}")

Balanced dataset: (15186, 35)
Label distribution:
label
0    7593
1    7593
Name: count, dtype: int64
Saved to: C:\Users\paulc\OneDrive\Projects\wimbledon-predictor\data\grass_features_balanced.csv
